## 01. 데이터 탐색 (EDA)

DuckDB 테이블 구조, NULL 패턴, 주소 형식, 좌표 매칭율을 분석합니다.

In [1]:
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가 (notebooks/ -> part09_trade_map/ -> projects/ -> 프로젝트 루트)
PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from projects.part09_trade_map.dashboard.db import execute_query

### 1. 테이블 목록 및 행 수

In [2]:
# 주요 테이블 행 수 확인
tables = ['매매', '전월세', '공동주택_전국', '좌표']
for t in tables:
    df = execute_query(f"SELECT COUNT(*) as cnt FROM {t}")
    print(f"{t}: {df.iloc[0]['cnt']:,}건")

매매: 189,759건
전월세: 1,064,392건
공동주택_전국: 9,458건
좌표: 8,094건


### 2. 매매 테이블 스키마 및 샘플

In [ ]:
# 매매 테이블 컬럼 구조
df = execute_query("DESCRIBE 매매")
display(df)

In [ ]:
# 매매 샘플 5건
df = execute_query("SELECT * FROM 매매 LIMIT 5")
display(df)

### 3. 시군구 컬럼 형식 분석

In [ ]:
# 시군구 컬럼의 단어 수별 분포
df = execute_query("""
    SELECT 
        len(regexp_split_to_array(regexp_replace(시군구, '\\s+', ' ', 'g'), ' ')) AS word_count,
        COUNT(*) AS cnt
    FROM 매매
    WHERE 시군구 IS NOT NULL
    GROUP BY word_count
    ORDER BY word_count
""")
display(df)

### 4. 좌표 테이블 매칭율 분석

In [ ]:
from src.utils.address import ROAD_ADDRESS_KEY_SQL

# 도로명주소_key 생성 후 좌표 매칭율 확인 (서울 2023년)
df = execute_query(f"""
    WITH keys AS (
        SELECT DISTINCT
            {ROAD_ADDRESS_KEY_SQL} AS 도로명주소_key
        FROM 매매
        WHERE 시군구 LIKE '서울특별시%'
            AND 계약년월 >= 202300
            AND 도로명 IS NOT NULL AND 도로명 != ''
    )
    SELECT
        COUNT(*) AS total_keys,
        COUNT(z.도로명주소) AS matched,
        ROUND(COUNT(z.도로명주소) * 100.0 / COUNT(*), 2) AS match_rate
    FROM keys k
    LEFT JOIN 좌표 z ON k.도로명주소_key = z.도로명주소
""")
display(df)

### 5. 공동주택_전국 테이블 분석

In [ ]:
# 단지구분코드별 건수 및 세대수
df = execute_query("""
    SELECT 
        단지구분코드,
        COUNT(*) AS cnt,
        SUM(세대수) AS total_세대수
    FROM 공동주택_전국
    GROUP BY 단지구분코드
    ORDER BY cnt DESC
""")
display(df)

In [ ]:
# 시도별 아파트 단지 수 및 세대수
df = execute_query("""
    SELECT 
        split_part(주소, ' ', 1) AS 시도,
        COUNT(*) AS 단지수,
        SUM(세대수) AS 총세대수
    FROM 공동주택_전국
    WHERE 단지구분코드 = '1'
    GROUP BY 시도
    ORDER BY 총세대수 DESC
""")
display(df)